In [1]:
import whisper

In [2]:
# base faster but less accurate
model = whisper.load_model("medium.en")

In [3]:
def extract_words(audio_path):
    """Transcribe audio and extract word-level timestamps.
    
    Args:
        audio_path (str): Path to the audio file.
    
    Returns:
        list: List of word dictionaries with timestamps.
    """
    result = model.transcribe(audio_path, word_timestamps=True)
    words = []
    for seg in result["segments"]:
        for w in seg["words"]:
            words.append(w)
    return words

In [ ]:
def count_fillers(words,    fillers = ["um", "uh", "like", "you know", "ah", "ahh", "oh", "hmm", "er", "mm", "mmm", "uhm", "huh", "eh", "uhhh", "so", "actually", "basically", "right", "well", "you see", "I mean", "kind of", "sort of"]):
    """
    Count total words and filler words in the transcript.\n
    aegs:        words (list): List of word dictionaries with 'word' key.
        fillers (list): List of filler words to count.
    Returns:
        tuple: (total_words, filler_count) 
    """
    total_words = len(words)
    filler_count = sum(1 for w in words if w["word"].lower() in fillers)
    return total_words, filler_count

In [16]:
import numpy as np

def _pause_quality_score(words, min_threshold=0.20, ideal_min=0.35, ideal_max=0.9, max_fail_threshold=1.2):
    """
    Calculate the percentage of pauses that fall within the ideal natural range (around 0.6s).
    Differentiates short articulatory stops, normal phrasing gaps, and long transition boundaries.
    
    Args:
        words (list): List of word dictionaries with 'start' and 'end' timestamps.
        min_threshold (float): Minimum gap to filter physical articulation/STT overlap (default: 0.20s).
        ideal_min (float): Lower bound of a natural phrase pause (default: 0.35s).
        ideal_max (float): Upper bound of a natural phrase pause (default: 0.9s).
        max_fail_threshold (float): Gaps larger than this are treated as natural slide/thought 
                                    transitions rather than sentence hesitations.
    
    Returns:
        tuple: (quality_score (float), ideal_pauses (list), real_pauses (list))
    """
    if not words or len(words) < 2:
        return 0.0, [], []
        
    real_pauses = []
    ideal_pauses = []
    
    for i in range(len(words) - 1):
        # Extract and explicitly force cast from np.float64 to native float
        current_end = float(words[i]['end'])
        next_start = float(words[i+1]['start'])
        
        gap = next_start - current_end
        
        # 1. Eliminate overlapping tokens, models glitches, and sub-threshold articulatory stops
        if gap < min_threshold:
            continue
            
        # 2. Check if the break is a massive structural transition (e.g., moving to a new section)
        # We track it as a real pause but can bypass severe sentence-level penalties if desired.
        real_pauses.append(round(gap, 4))
        
        # 3. Evaluate if it falls squarely within the natural phrase "comfort zone"
        if ideal_min <= gap <= ideal_max:
            ideal_pauses.append(round(gap, 4))
        elif gap > max_fail_threshold:
            # OPTIONAL: You can choose to append these to ideal_pauses as well if you don't
            # want to penalize presentation transitions. Currently, it penalizes 0.8s to 1.2s gaps.
            pass
                
    if not real_pauses:
        return 0.0, [], []
        
    # 4. Calculate final presentation pacing accuracy
    quality_score = (len(ideal_pauses) / len(real_pauses)) * 100
    
    return round(quality_score, 2), ideal_pauses, real_pauses

In [17]:
from moviepy import VideoFileClip

#video = r"D:\Hossam\study\Graduation project\AI-Video-interview-analysis-\DataSet\man3.mp4"
extracted_audio_path = r"D:\Hossam\study\Graduation project\AI-Video-interview-analysis-\DataSet\man3.wav"
# Extract audio from video
#clip = VideoFileClip(video)
#clip.audio.write_audiofile(extracted_audio_path)
words = extract_words(extracted_audio_path)
print("Words:", words)
quality_score, ideal_pauses, real_pauses = _pause_quality_score(words)
print("Pause Rate:", quality_score)
print("Ideal Pauses:", ideal_pauses)
print("Real Pauses:", real_pauses)

Words: [{'word': ' For', 'start': np.float64(0.0), 'end': np.float64(0.4), 'probability': np.float64(0.9595277309417725)}, {'word': ' me,', 'start': np.float64(0.4), 'end': np.float64(0.7), 'probability': np.float64(0.998776376247406)}, {'word': ' New', 'start': np.float64(0.98), 'end': np.float64(0.98), 'probability': np.float64(0.6196600198745728)}, {'word': ' Zealand', 'start': np.float64(0.98), 'end': np.float64(1.1), 'probability': np.float64(0.9948939085006714)}, {'word': ' has', 'start': np.float64(1.1), 'end': np.float64(1.44), 'probability': np.float64(0.9512558579444885)}, {'word': ' and', 'start': np.float64(1.44), 'end': np.float64(1.64), 'probability': np.float64(0.8500655889511108)}, {'word': ' always', 'start': np.float64(1.64), 'end': np.float64(1.88), 'probability': np.float64(0.9567802548408508)}, {'word': ' will', 'start': np.float64(1.88), 'end': np.float64(2.1), 'probability': np.float64(0.9973192811012268)}, {'word': ' be', 'start': np.float64(2.1), 'end': np.floa

In [12]:
from moviepy import VideoFileClip

video = r"D:\Hossam\study\Graduation project\AI-Video-interview-analysis-\DataSet\Hossam_video.mp4"
extracted_audio_path = r"D:\Hossam\study\Graduation project\AI-Video-interview-analysis-\DataSet\Hossam_audio.wav"
# Extract audio from video
clip = VideoFileClip(video)
clip.audio.write_audiofile(extracted_audio_path)
words = extract_words(extracted_audio_path)
print("Words:", words)
quality_score, ideal_pauses, real_pauses = _pause_quality_score(words)
print("Pause Rate:", quality_score)
print("Ideal Pauses:", ideal_pauses)
print("Real Pauses:", real_pauses)

MoviePy - Writing audio in D:\Hossam\study\Graduation project\AI-Video-interview-analysis-\DataSet\Hossam_audio.wav


MoviePy - Done.
Words: [{'word': ' Hello,', 'start': np.float64(0.0), 'end': np.float64(0.36), 'probability': np.float64(0.37579345703125)}, {'word': ' my', 'start': np.float64(0.42), 'end': np.float64(0.54), 'probability': np.float64(0.9226248264312744)}, {'word': ' name', 'start': np.float64(0.54), 'end': np.float64(0.68), 'probability': np.float64(0.9961979985237122)}, {'word': ' is', 'start': np.float64(0.68), 'end': np.float64(0.8), 'probability': np.float64(0.9871507883071899)}, {'word': ' Isfasem', 'start': np.float64(0.8), 'end': np.float64(1.12), 'probability': np.float64(0.2973587540909648)}, {'word': ' Abadim.', 'start': np.float64(1.12), 'end': np.float64(1.46), 'probability': np.float64(0.6860355337460836)}, {'word': ' I', 'start': np.float64(1.56), 'end': np.float64(1.62), 'probability': np.float64(0.9819701910018921)}, {'word': ' am', 'start': np.float64(1.62), 'end': np.float64(1.76), 'probability': np.float64(0.8099180459976196)}, {'word': ' graduate', 'start': np.floa

In [14]:
from moviepy import VideoFileClip

video = r"D:\Hossam\study\Graduation project\AI-Video-interview-analysis-\DataSet\woman.mp4"
extracted_audio_path = r"D:\Hossam\study\Graduation project\AI-Video-interview-analysis-\DataSet\woman_audio.wav"
# Extract audio from video
clip = VideoFileClip(video)
clip.audio.write_audiofile(extracted_audio_path)
words = extract_words(extracted_audio_path)
print("Words:", words)
quality_score, ideal_pauses, real_pauses = _pause_quality_score(words)
print("Pause Rate:", quality_score)
print("Ideal Pauses:", ideal_pauses)
print("Real Pauses:", real_pauses)

MoviePy - Writing audio in D:\Hossam\study\Graduation project\AI-Video-interview-analysis-\DataSet\woman_audio.wav


MoviePy - Done.
Words: [{'word': ' Hello,', 'start': np.float64(0.0), 'end': np.float64(1.04), 'probability': np.float64(0.8264926671981812)}, {'word': ' my', 'start': np.float64(1.16), 'end': np.float64(1.36), 'probability': np.float64(0.9874861836433411)}, {'word': ' name', 'start': np.float64(1.36), 'end': np.float64(1.52), 'probability': np.float64(0.9991146922111511)}, {'word': ' is', 'start': np.float64(1.52), 'end': np.float64(1.72), 'probability': np.float64(0.9984808564186096)}, {'word': ' Brenna', 'start': np.float64(1.72), 'end': np.float64(2.06), 'probability': np.float64(0.8721128106117249)}, {'word': ' Harrity', 'start': np.float64(2.06), 'end': np.float64(2.36), 'probability': np.float64(0.19520193338394165)}, {'word': ' and', 'start': np.float64(2.36), 'end': np.float64(2.88), 'probability': np.float64(0.6855847835540771)}, {'word': " I'm", 'start': np.float64(2.88), 'end': np.float64(3.16), 'probability': np.float64(0.9701374173164368)}, {'word': ' applying', 'start': 